# Stream batches from a queue, then upsert into Qdrant
The fake producer uses ordered messages from one partition. No broker or credentials are needed.
A real application can use `confluent-kafka.Consumer`, disable automatic commits, and commit the next offset only after durability.
Track offsets separately for each topic and partition. The row-to-offset mapping here assumes contiguous offsets in one partition.
After a crash, reconcile broker offsets with the durable local store prefix before replay. The store prefix wins if the broker commit lagged.
Kafka transactions do not include a local ShardSink. This example shows this reconciliation, not a distributed transaction.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
import os
import platform
from datetime import date

scratch = TemporaryDirectory(prefix="fse-recipe-")
work = Path(scratch.name)
# Third-party caches also belong to this temporary directory.
for name in ("HF_HOME", "XDG_CACHE_HOME", "NUMBA_CACHE_DIR", "MPLCONFIGDIR"):
    os.environ[name] = str(work / name.lower())
os.environ["ANONYMIZED_TELEMETRY"] = "False"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
import numpy as np
import fse
print(date.today().isoformat(), platform.platform(), platform.machine())

In [ ]:
# Deliberately tiny, hand-built word rows. These demonstrate APIs, not model quality.
words = ["cat", "dog", "pet", "kitten", "puppy", "car", "bus", "road", "train", "truck"]
vectors = np.array([
    [1, 0, .2], [1, .1, 0], [1, 0, 0], [1, 0, .3], [1, .2, 0],
    [0, 1, .2], [0, 1, .1], [0, 1, 0], [0, 1, .3], [0, 1, .4],
], dtype=np.float32)
encoder = fse.Encoder.from_arrays(vectors, words=words, normalize=False)
texts = ["cat pet", "dog puppy", "car road", "bus truck"]

In [ ]:
messages = texts * 3
broker_next = 0
queue_store = work / "queue-store"

def consume(interrupt=False):
    global broker_next
    with fse.ShardSink(queue_store, encoder, shard_rows=2) as sink:
        start = sink.first_row
        if broker_next > start:
            raise RuntimeError("broker offset exceeds durable store prefix")
        # Reconcile a broker commit that lagged publication, without replaying stored rows.
        broker_next = start
        polls = [messages[i:i+2] for i in range(start, len(messages), 2)]
        stream = encoder.encode_batches(polls, sink=sink, workers=1)
        try:
            for batch in stream:
                assert batch.durable_rows is not None
                if interrupt:
                    # Simulate failure after durable publication and before broker commit.
                    break
                broker_next = start + batch.durable_rows
        finally:
            stream.close()
        if not interrupt:
            broker_next = start + sink.flush()
consume(interrupt=True)
assert broker_next == 0
consume()
consume()
with fse.VectorStore.open(queue_store) as store:
    assert len(store) == broker_next == len(messages)
print("Durable rows and next Kafka offset:", broker_next)

## Qdrant stays in the application
[Qdrant's Python client](https://github.com/qdrant/qdrant-client) supports local in-memory mode.
Upsert batches with stable integer IDs. Replay replaces the same points instead of creating duplicates.
In-memory Qdrant is not durable. Do not acknowledge a production queue on the strength of this demonstration.
For a remote server, configure its acknowledgement and replication policy separately.

In [ ]:
from qdrant_client import QdrantClient, models
client = QdrantClient(":memory:")
client.create_collection("recipe", vectors_config=models.VectorParams(size=encoder.output_dim, distance=models.Distance.COSINE))
for _ in range(2):  # replay the exact same IDs
    for batch in encoder.encode_batches([texts[:2], texts[2:]], workers=1):
        client.upsert("recipe", points=[models.PointStruct(id=batch.first_row+i, vector=row.tolist()) for i, row in enumerate(batch.rows)], wait=True)
assert client.count("recipe", exact=True).count == len(texts)
print("Qdrant points after replay:", client.count("recipe", exact=True).count)
client.close()

In [ ]:
scratch.cleanup()